# Currentness and timeliness measures

ISO/IEC 5259-2 `Cur-ML-*` and ISO/IEC 25024 `Cur-I-*`/`Tml-*` measures: whether temporal data is fresh enough, updated often enough, arrived on time. 

All pin `reference_time` here for reproducible output, the default is the wall clock.

In [1]:
from datetime import datetime, timedelta

import polars as pl

from dqmeasure import (
    FeatureCurrentness,
    RecordCurrentness,
    TimelinessOfDataItems,
    TimelinessOfUpdate,
    UpdateFrequency,
)

now = datetime(2026, 1, 1)

## FeatureCurrentness
**"Is the cell's age within the required range?"**

Age = `reference_time - timestamp`. `fit` learns `[min_age, max_age]` from the ages observed in clean data.

In [2]:
clean = pl.DataFrame({"updated_at": [now - timedelta(days=d) for d in [1, 2, 3, 5]]})
dirty = pl.DataFrame({"updated_at": [now - timedelta(days=d) for d in [1, 4, 30, 400]]})

measure = FeatureCurrentness("updated_at", reference_time=now).fit(clean)
measure.min_age_, measure.max_age_

(datetime.timedelta(days=1), datetime.timedelta(days=5))

In [3]:
measure.predict(dirty)

updated_at
f64
1.0
1.0
0.0
0.0


In [4]:
measure.score(dirty)

0.5

## RecordCurrentness
**"Is every temporal cell in the record within its required age range?"**

Table-scoped: one age range per temporal column, learned independently.

In [5]:
clean = pl.DataFrame(
    {
        "created_at": [now - timedelta(days=d) for d in [10, 12, 15]],
        "updated_at": [now - timedelta(days=d) for d in [1, 2, 1]],
    }
)
dirty = pl.DataFrame(
    {
        "created_at": [now - timedelta(days=d) for d in [11, 12, 400]],
        "updated_at": [now - timedelta(days=d) for d in [1, 60, 1]],
    }
)

measure = RecordCurrentness(reference_time=now).fit(clean)
measure.predict(dirty)

record
f64
1.0
0.0
0.0


In [6]:
measure.score(dirty)

0.3333333333333333

Row 3's `created_at` and row 2's `updated_at` each break their column's learned range.

## UpdateFrequency
**"Did this update arrive within the required interval of the one before it?"**

Unit = update event, not cell. `fit` learns the largest gap between consecutive clean events; row order doesn't matter, events are sorted by timestamp internally.

In [7]:
clean = pl.DataFrame({"ts": [now + timedelta(minutes=m) for m in [0, 1, 2, 3, 4]]})
dirty = pl.DataFrame({"ts": [now + timedelta(minutes=m) for m in [0, 1, 2, 10, 11]]})

measure = UpdateFrequency("ts").fit(clean)
measure.max_interval_

datetime.timedelta(seconds=60)

In [8]:
measure.predict(dirty)

ts
f64
null
1.0
1.0
0.0
1.0


In [9]:
measure.score(dirty)

0.75

## TimelinessOfDataItems
**"Did the data become available within the required latency of the event it records?"**

Two columns: `column` (availability) and `event_column` (when the phenomenon happened).

In [10]:
clean = pl.DataFrame(
    {
        "event_at": [now - timedelta(hours=h) for h in [5, 4, 3]],
        "available_at": [now - timedelta(hours=h) for h in [4, 3, 2]],  # 1h latency each
    }
)
dirty = pl.DataFrame(
    {
        "event_at": [now - timedelta(hours=h) for h in [5, 4, 3]],
        "available_at": [now - timedelta(hours=4), now, None],
    }
)  # row 2 arrives 4h after its event (3h over the 1h budget), row 3 never arrived

measure = TimelinessOfDataItems("available_at", "event_at").fit(clean)
measure.max_latency_

datetime.timedelta(seconds=3600)

In [11]:
measure.predict(dirty)

available_at
f64
1.0
0.0
0.0


In [12]:
measure.score(dirty)

0.3333333333333333

## TimelinessOfUpdate
**"Did the update land within the SLA of when it was due?"**

Same structure as `TimelinessOfDataItems`, relabeled: `due_column` names when an update was owed, not when something happened.

In [13]:
clean = pl.DataFrame(
    {
        "due_at": [now - timedelta(hours=h) for h in [5, 4, 3]],
        "updated_at": [now - timedelta(hours=h) for h in [4, 3, 2]],
    }
)
dirty = pl.DataFrame(
    {
        "due_at": [now - timedelta(hours=h) for h in [5, 4, 3]],
        "updated_at": [now - timedelta(hours=4), now, None],
    }
)

measure = TimelinessOfUpdate("updated_at", "due_at").fit(clean)
measure.predict(dirty)

updated_at
f64
1.0
0.0
0.0


In [14]:
measure.score(dirty)

0.3333333333333333